<a href="https://colab.research.google.com/github/omnateeta/Generative-AI-Lab/blob/main/GenAiprogram09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel
from typing import List, Optional

In [39]:
class InstitutionDetails(BaseModel):
   founder: Optional[str]
   founded: Optional[str]
   branches: Optional[List[str]]
   employees: Optional[int]
   summary: Optional[str]

In [41]:
import re

def fetch_institution_data(institution_name: str) -> InstitutionDetails:
    formatted_name = institution_name.replace(" ", "_")
    url = f"https://en.wikipedia.org/wiki/{formatted_name}"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch Wikipedia page for {institution_name}. HTTP Status code: {response.status_code}")

    soup = BeautifulSoup(response.text, 'html.parser')

    infobox = soup.find('table', {'class': 'infobox'})

    founder = None
    founded = None
    branches = []
    employees = None
    summary = ""

    if infobox:
        rows = infobox.find_all('tr')
        for row in rows:
            header = row.find('th')
            data = row.find('td')

            if header and data:
                header_text = header.get_text(strip=True).lower()
                data_text = data.get_text(strip=True)

                if 'founder' in header_text or 'founders' in header_text:
                    founder = data_text
                elif 'founded' in header_text or 'established' in header_text:
                    founded = data_text
                elif 'employees' in header_text or 'staff' in header_text or 'total staff' in header_text or 'academic staff' in header_text:
                    numbers = re.findall(r'\d[\d,]*', data_text)
                    if numbers:
                        try:
                            employees = int(numbers[0].replace(',', ''))
                        except ValueError:
                            employees = None
                elif 'branches' in header_text or 'campuses' in header_text or 'locations' in header_text or 'affiliations' in header_text:
                    branches = [branch.strip() for branch in data_text.split(';') if branch.strip()]

    paragraphs = soup.find_all('p')
    current_summary_parts = []
    for p in paragraphs:
        paragraph_text = p.get_text(strip=True)
        if paragraph_text:
            if any(term in paragraph_text.lower() for term in ["for other uses, see", "coordinates:", "this article is about"]):
                continue
            if p.find('img'):
                continue

            current_summary_parts.append(paragraph_text)
            if len(" ".join(current_summary_parts)) > 500 and len(current_summary_parts) > 2:
                break

    summary = " ".join(current_summary_parts)
    if len(summary) > 700:
        summary = summary[:700] + "..."

    return InstitutionDetails(
        founder=founder,
        founded=founded,
        branches=branches,
        employees=employees,
        summary=summary
    )

In [42]:
def main():
    institution_name = input("Enter the name of the institution: ")

    try:
        institution_details = fetch_institution_data(institution_name)
        print(f"\nDetails of {institution_name}:\n")
        print(f"Founder: {institution_details.founder}")
        print(f"Founded: {institution_details.founded}")
        print(f"Branches: {', '.join(institution_details.branches) if institution_details.branches else 'N/A'}")
        print(f"Employees: {institution_details.employees if institution_details.employees else 'N/A'}")
        print(f"Summary: {institution_details.summary}")
    except Exception as e:
        print(f"Error: {e}")

In [44]:
if __name__ == "__main__":
    main()

Enter the name of the institution: Stanford University 

Details of Stanford University :

Founder: LelandandJane Stanford
Founded: 1885
Branches: NCAA Division I FBS–ACCIRAPCCSCMPSF
Employees: 18369
Summary: Leland Stanford Junior University,[9][10]commonly referred to asStanford University, is aprivateresearch universityinStanford, California, United States. It was founded in 1885 by railroad magnateLeland Stanford(the eighthgovernorof and then-incumbentUnited States senator representing California) and his wife,Jane, in memory of their only child,Leland Jr.[11] The university admitted its first students in 1891,[11][12]opening as acoeducationalandnon-denominationalinstitution. It struggled financially after Leland died in 1893 and again after much of the campus was damaged by the1906 San Francisco earthquake.[13]FollowingWorld War II, universityprovostFrederick Termaninspired anentrepreneurial...
